In [1]:
import numpy as np
import matplotlib.pyplot as plt
import re
import math
import random

In [2]:
with open('some_words.txt' , 'r' , encoding='utf-8') as f:
    corpus = f.read()
    corpus = re.sub(r'[^a-z\s]', ' ', corpus.lower())
words = corpus.lower().split()
words[:10]

['alice',
 'was',
 'beginning',
 'to',
 'get',
 'very',
 'tired',
 'of',
 'sitting',
 'by']

In [3]:
vocab = sorted(list(set(words))) #-> unique words hai 
word_idx = {w:i for i , w in enumerate(vocab)}
idx_word = {i:w for i , w in enumerate(vocab)}

In [4]:
# creating training pairs of consecutive words
X_data = [word_idx[words[i]] for i in range(len(words) - 1)] # -> indexes ko nikaal rahe hai.
Y_data = [word_idx[words[i + 1]] for i in range(len(words) - 1)] # -> indexes main se 1 kam kar diya 

In [5]:
V = len(vocab)

In [6]:
import torch
import torch.nn as nn

In [7]:
class ContextMLP(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, block_size=3, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        # Flatten concatenated context embeddings: block_size * emb_dim
        self.fc1 = nn.Linear(block_size * emb_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x shape: (batch_size, block_size)
        h = self.emb(x).view(x.size(0), -1) # Flatten context embeddings
        h = self.act(self.fc1(h))
        logits = self.fc2(h)
        return logits

model = ContextMLP(vocab_size=V, emb_dim=64, block_size=3, hidden_dim=128)

In [8]:
# Context size of 3 words instead of 1
block_size = 3
X_data, Y_data = [], []

for i in range(len(words) - block_size):
    X_data.append([word_idx[w] for w in words[i:i+block_size]])
    Y_data.append(word_idx[words[i+block_size]])

X_data = torch.tensor(X_data) # Shape: (N, 3)
Y_data = torch.tensor(Y_data)

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [10]:
epochs = 300

for epoch in range(epochs):
    # 1. Forward pass (processes all samples at once)
    logits = model(X_data)
    loss = criterion(logits, Y_data)

    # 2. Backward pass
    optimizer.zero_grad()
    loss.backward()
    
    # 3. Update weights
    optimizer.step()

    # Print progress
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss (NLL): {loss.item():.4f}")

Epoch   1/300 | Loss (NLL): 7.8688
Epoch  20/300 | Loss (NLL): 4.0330
Epoch  40/300 | Loss (NLL): 2.1429
Epoch  60/300 | Loss (NLL): 1.2926
Epoch  80/300 | Loss (NLL): 0.7992
Epoch 100/300 | Loss (NLL): 0.5079
Epoch 120/300 | Loss (NLL): 0.3463
Epoch 140/300 | Loss (NLL): 0.2684
Epoch 160/300 | Loss (NLL): 0.2358
Epoch 180/300 | Loss (NLL): 0.2230
Epoch 200/300 | Loss (NLL): 0.2168
Epoch 220/300 | Loss (NLL): 0.2138
Epoch 240/300 | Loss (NLL): 0.2120
Epoch 260/300 | Loss (NLL): 0.2110
Epoch 280/300 | Loss (NLL): 0.2103
Epoch 300/300 | Loss (NLL): 0.2096


In [11]:
@torch.no_grad()
def predict_next_word(prompt, model, word_idx, idx_word, block_size=3):
    model.eval()
    words = prompt.lower().split()[-block_size:]
    
    # Convert input words to index tensor
    x = torch.tensor([[word_idx[w] for w in words]]) # Shape: (1, block_size)
    
    # Get probabilities
    logits = model(x)
    probs = torch.softmax(logits, dim=-1)
    
    # Pick top prediction
    top_idx = torch.argmax(probs, dim=-1).item()
    top_prob = probs[0, top_idx].item()
    
    return idx_word[top_idx], top_prob

# Example usage (adjust context to match your block_size):
prompt = "alice was beginning" 
next_word, confidence = predict_next_word(prompt, model, word_idx, idx_word, block_size=3)
print(f"Prompt: '{prompt}' -> Next Word: '{next_word}' (Confidence: {confidence*100:.2f}%)")

Prompt: 'alice was beginning' -> Next Word: 'to' (Confidence: 57.25%)


In [12]:
@torch.no_grad()
def generate_text(prompt, model, word_idx, idx_word, block_size=3, max_new_tokens=15, temperature=1.0):
    model.eval()
    tokens = prompt.lower().split()
    
    for _ in range(max_new_tokens):
        # Take the most recent context tokens
        context = tokens[-block_size:]
        x = torch.tensor([[word_idx[w] for w in context]])
        
        # Calculate probabilities with temperature scaling
        logits = model(x) / temperature
        probs = torch.softmax(logits, dim=-1)
        
        # Sample next token from probability distribution
        next_idx = torch.multinomial(probs, num_samples=1).item()
        tokens.append(idx_word[next_idx])
        
    return " ".join(tokens)

# Example usage:
generated_story = generate_text(
    prompt="alice was beginning", 
    model=model, 
    word_idx=word_idx, 
    idx_word=idx_word, 
    block_size=3, 
    max_new_tokens=20,
    temperature=0.8 # Lower = more deterministic, Higher = more creative/random
)

print(generated_story)

alice was beginning very angrily but the hatter was the only one who got any advantage from the change and alice was soon
